# SHEAR v3: SmolVLA fine-tune on a Colab T4

Trains SmolVLA (`lerobot/smolvla_base`) on the SHEAR v1-rig dataset (base camera + claw camera, scripted cuts): the **action expert and the vision encoder** train, the language layers stay frozen. Same recipe as `policy/train_smolvla_lora.sh v3`, moved to Colab's 16 GB T4.

**Before you run**
1. Runtime → Change runtime type → **T4 GPU**.
2. Colab **Secrets** (key icon on the left): add `HF_TOKEN` = a Hugging Face token with *write* access, and switch on "Notebook access". The dataset and the checkpoints are private repos on the Hub.
3. Run the cells top to bottom. Keep the tab open: the free tier ends a session after a while.

**If the session dies**, run the cells again from the top. The train cell finds the last checkpoint on the Hub and resumes from it (same step counter, same optimizer state, same data order). Nothing is lost except the steps since the last checkpoint (at most `SAVE_FREQ`).

In [ ]:
# 1. The GPU this session got. A T4 has 16 GB; the laptop's RTX 1000 Ada has 6.
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# 2. lerobot, pinned to the SAME commit as the laptop's fork (0.6.1 + 70, 2026-11), with the SmolVLA and video extras.
#    Takes a few minutes. If a later cell fails on an import, Runtime → Restart session, then continue from cell 3.
%pip install -q "lerobot[smolvla,av-dep] @ git+https://github.com/huggingface/lerobot@5aa74557"
import lerobot, torch
print("lerobot", lerobot.__version__, "torch", torch.__version__, "cuda", torch.cuda.is_available())

In [ ]:
# 3. The SHEAR patch (same as the laptop's local lerobot edit, K7 2026-11-06).
#    Upstream: train_expert_only freezes the WHOLE vision-language model, vision encoder included, so
#    --policy.freeze_vision_encoder=false does nothing with it. After this patch the vision flag wins: the vision
#    encoder trains together with the action expert, the language layers stay frozen (186M of 450M params train).
import pathlib
import lerobot.policies.smolvla.smolvlm_with_expert as m

p = pathlib.Path(m.__file__)
s = p.read_text()
MARK = "# SHEAR patch: the vision encoder trains with the expert"
if MARK in s:
    print("already patched:", p)
else:
    a = ("        # To avoid unused params issue with distributed training\n"
         "        for name, params in self.lm_expert.named_parameters():\n")
    b = ("            self.get_vlm_model().vision_model.eval()\n\n"
         "        if self.train_expert_only:\n            self.vlm.eval()\n")
    assert s.count(a) == 1 and s.count(b) == 1, "the file changed, anchors not found: check the lerobot commit"
    s = s.replace(a, f"        {MARK}\n"
                     "        if self.train_expert_only and not self.freeze_vision_encoder:\n"
                     "            self.get_vlm_model().vision_model.train()\n"
                     "            for params in self.get_vlm_model().vision_model.parameters():\n"
                     "                params.requires_grad = True\n" + a)
    s = s.replace(b, b + "            if not self.freeze_vision_encoder:\n"
                         "                self.get_vlm_model().vision_model.train(mode)\n")
    p.write_text(s)
    print("patched:", p)

In [ ]:
# 4. Hugging Face login + the run's settings.
import os
from google.colab import userdata
from huggingface_hub import whoami

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("logged in as", whoami()["name"])

DATASET = "krish5831/shear_sim_scripted_v1"      # 200 scripted episodes on the v1 rig (private)
MODEL_REPO = "krish5831/shear_smolvla_v3"         # every checkpoint lands here under checkpoints/<step>/ (private)
# SmolVLA's base expects its own camera names. The base camera is the overview, the claw camera the wrist view.
# The SAME map goes to every rollout of the trained policy (sim/rollout_policy.py does this).
RENAME = '{"observation.images.base": "observation.images.camera1", "observation.images.claw": "observation.images.camera2"}'
STEPS = 100_000        # K7's call. lerobot counts every batch as a step: 100k x 8 = 800k samples, ~5.6 passes over the data.
BATCH, ACCUM = 8, 4    # 8 per batch on the T4, gradients summed over 4 batches = effective batch 32, as on the laptop
SAVE_FREQ = 5_000      # a checkpoint on the Hub every 5k steps: what a dead session costs at most
PRECISION = "fp16"     # the T4 is fast in fp16 (no bf16 hardware). If the smoke prints loss=nan, set "no" (slower).

COMMON = [
    f"--policy.path=lerobot/smolvla_base", f"--dataset.repo_id={DATASET}", "--dataset.video_backend=pyav",
    "--policy.device=cuda", f"--rename_map={RENAME}", "--wandb.enable=false",
    "--policy.freeze_vision_encoder=false", f"--batch_size={BATCH}", f"--accelerator.gradient_accumulation.steps={ACCUM}",
    f"--accelerator.mixed_precision={PRECISION}", "--num_workers=2",
]

In [ ]:
# 5. Smoke: 30 steps. Downloads the dataset (3.3 GB) and the base model, checks the loss is a number,
#    and prints the time per step -> the hours the full run needs. Nothing goes to the Hub from this cell.
import re, subprocess, shutil, time

shutil.rmtree("/content/smoke", ignore_errors=True)
cmd = ["lerobot-train", *COMMON, "--steps=30", "--save_freq=30", "--log_freq=10",
       "--policy.push_to_hub=false", "--output_dir=/content/smoke", "--job_name=smoke"]
log = []
with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as pr:
    for line in pr.stdout:
        print(line, end=""); log.append(line)
assert pr.returncode == 0, "the smoke failed: read the lines above"
steps = [l for l in log if "updt_s:" in l]
loss = re.search(r"loss:([\d.na]+)", steps[-1]).group(1)
updt = float(re.search(r"updt_s:([\d.]+)", steps[-1]).group(1))
data = float(re.search(r"data_s:([\d.]+)", steps[-1]).group(1))
print(f"\nloss {loss}   {updt:.2f} s per step + {data:.2f} s loading   ->   {STEPS} steps = {STEPS*(updt+data)/3600:.1f} h")
if loss == "nan":
    print("!! loss is nan: set PRECISION = 'no' in cell 4 and run cells 4 and 5 again")

In [ ]:
# 6. The run. Fresh if the Hub repo holds no checkpoint yet, otherwise RESUMED from the newest one.
#    Every SAVE_FREQ steps a checkpoint goes to MODEL_REPO (weights 0.9 GB + optimizer state ~1.5 GB).
#    Run this cell again after any disconnect.
import datetime as dt, subprocess
from huggingface_hub import HfApi

api = HfApi()
try:
    have = [f for f in api.list_repo_files(MODEL_REPO) if f.startswith("checkpoints/")]
except Exception:
    have = []
stamp = dt.datetime.now().strftime("%m%d_%H%M%S")
if have:
    newest = max(f.split("/")[1] for f in have)
    print("resuming from", MODEL_REPO, "checkpoint", newest)
    cmd = ["lerobot-train", "--resume=true", f"--config_path={MODEL_REPO}", f"--output_dir=/content/train/v3_{stamp}"]
else:
    print("fresh run ->", MODEL_REPO)
    cmd = ["lerobot-train", *COMMON, f"--steps={STEPS}", f"--save_freq={SAVE_FREQ}", "--log_freq=50",
           "--save_checkpoint_to_hub=true", "--policy.push_to_hub=true", f"--policy.repo_id={MODEL_REPO}",
           "--policy.private=true", f"--output_dir=/content/train/v3_{stamp}", "--job_name=shear_smolvla_v3"]
subprocess.run(cmd, check=True)

In [ ]:
# 7. What is on the Hub now.
from huggingface_hub import HfApi
files = HfApi().list_repo_files(MODEL_REPO)
print(sorted({f.split("/")[1] for f in files if f.startswith("checkpoints/")}))

## Picking a checkpoint

The loss says little about cuts (Dream Machines' π0.5 write-up: pick checkpoints by rollouts, not loss). Each checkpoint is rolled out in the sim on the laptop, on rows the policy never saw:

```bash
# on the laptop, lerobot env, S.H.E.A.R root
hf download krish5831/shear_smolvla_v3 --include "checkpoints/020000/pretrained_model/*" --local-dir outputs/train/shear_smolvla_v3_hub
python sim/rollout_policy.py --ckpt outputs/train/shear_smolvla_v3_hub/checkpoints/020000/pretrained_model --rtc 10 --rtc-horizon 20 --smooth 0.5 --windowed
```

Stop the Colab run when two checkpoints in a row cut no better than the one before.